## Prune-Distill Pipelines & Subspace-Targeted Distillation

## Combined Recipe — Prune 30% → Distill to Recover Accuracy

**Pruning** and **distillation** are complementary techniques:
* **Pruning** shrinks the architecture to reduce latency and VRAM footprint
* **Distillation** restores lost accuracy by transferring dark knowledge back into the pruned capacity

### 1. Why Simple Fine-Tuning Fails After Pruning

When 30% of a model's channels or attention heads are pruned, fine-tuning with standard cross-entropy on ground-truth targets ($y^*$) often results in sub-optimal recovery:

* **Ground-truth labels** provide hard targets ($[0, 0, 1, 0]$), forcing the capacity-constrained student to overfit.
* **Soft probabilities** from the unpruned Teacher model ($p_{\text{Teacher}}$) provide explicit class correlations, guiding the pruned Student ($q_{\text{Student}}$) along smoother optimization trajectories.

### 2. The 3-Stage Prune-Distill Pipeline

```
 ┌─────────────────┐         1. Structured Pruning          ┌─────────────────┐
 │   Base Model    │ ─────────────────────────────────────► │  Pruned Model   │
 │ (100% Capacity) │   Remove 30% Heads/MLP Channels        │  (70% Capacity) │
 └─────────────────┘                                        └─────────────────┘
          │                                                          │
          │ Frozen Teacher                                           │ Trainable Student
          ▼                                                          ▼
 ┌─────────────────┐                                        ┌─────────────────┐
 │  Teacher Model  │ ───► 2. On-Policy / Soft Logit KL ───► │  Student Model  │
 │ (100% Capacity) │      Transfer Dark Knowledge           │ (70% Capacity)  │
 └─────────────────┘                                        └─────────────────┘
                                                                     │
                                                                     ▼ 3. Accuracy Recovery
                                                            ┌─────────────────┐
                                                            │ Recovery Target │
                                                            │  (~98-100% Acc) │
                                                            └─────────────────┘
```